In [1]:
import numpy as np
import pandas as pd

np.random.seed(42)

N_UNITS = 400      # size of our simulated fleet
MAX_DAYS = 500     # max days we observe each unit

rows = []

for unit_id in range(1, N_UNITS + 1):
    # Decide up front: will this unit experience a failure during our observation window?
    will_fail = np.random.rand() < 0.18   # ~18% of units fail -- failures should be the minority class
    lifespan = np.random.randint(120, MAX_DAYS) if will_fail else MAX_DAYS

    # Each unit has its own "baseline" healthy sensor signature (unit-to-unit variation)
    base_vibration = np.random.normal(2.0, 0.3)
    base_coolant_pressure = np.random.normal(140, 8)
    base_compressor_temp = np.random.normal(65, 4)

    for day in range(lifespan):
        if will_fail:
            days_to_failure = lifespan - day
            # degradation ramps up only in the last 60 days before failure
            degradation = max(0, 1 - (days_to_failure / 60)) if days_to_failure < 60 else 0
        else:
            days_to_failure = -1
            degradation = 0

        vibration = base_vibration + degradation * np.random.uniform(1.5, 3.0) + np.random.normal(0, 0.15)
        coolant_pressure = base_coolant_pressure - degradation * np.random.uniform(15, 30) + np.random.normal(0, 3)
        compressor_temp = base_compressor_temp + degradation * np.random.uniform(10, 25) + np.random.normal(0, 2)
        cargo_temp_deviation = abs(np.random.normal(0, 0.5)) + degradation * np.random.uniform(1, 3)

        failure_within_7d = 1 if (will_fail and 0 < days_to_failure <= 7) else 0

        rows.append({
            "unit_id": unit_id,
            "day": day,
            "vibration_mm_s": round(vibration, 3),
            "coolant_pressure_kpa": round(coolant_pressure, 2),
            "compressor_temp_c": round(compressor_temp, 2),
            "cargo_temp_deviation_c": round(cargo_temp_deviation, 3),
            "failure_within_7d": failure_within_7d,
        })

df = pd.DataFrame(rows)
print(df.shape)
print(df["failure_within_7d"].value_counts())
df.head()

(186102, 7)
failure_within_7d
0    185612
1       490
Name: count, dtype: int64


,unit_id,day,vibration_mm_s,coolant_pressure_kpa,compressor_temp_c,cargo_temp_deviation_c,failure_within_7d
0,1,0,1.818,140.81,65.07,0.121,0
1,1,1,1.379,141.27,64.63,0.733,0
2,1,2,1.633,142.92,65.09,0.300,0
3,1,3,1.623,140.95,66.11,0.411,0
4,1,4,1.483,143.21,67.88,0.058,0


In [2]:
sensor_cols = ["vibration_mm_s", "coolant_pressure_kpa", "compressor_temp_c", "cargo_temp_deviation_c"]
WINDOW = 7  # 7-day rolling window

df = df.sort_values(["unit_id", "day"]).reset_index(drop=True)

feature_frames = []
for unit_id, g in df.groupby("unit_id"):
    g = g.copy()
    for col in sensor_cols:
        g[f"{col}_roll_mean"] = g[col].rolling(WINDOW, min_periods=1).mean()
        g[f"{col}_roll_std"] = g[col].rolling(WINDOW, min_periods=1).std().fillna(0)
        g[f"{col}_trend"] = g[col] - g[col].shift(WINDOW).fillna(g[col])
    feature_frames.append(g)

feat_df = pd.concat(feature_frames, ignore_index=True)
print(feat_df.shape)
feat_df.head()

(186102, 19)


,unit_id,day,vibration_mm_s,coolant_pressure_kpa,compressor_temp_c,cargo_temp_deviation_c,failure_within_7d,vibration_mm_s_roll_mean,vibration_mm_s_roll_std,vibration_mm_s_trend,coolant_pressure_kpa_roll_mean,coolant_pressure_kpa_roll_std,coolant_pressure_kpa_trend,compressor_temp_c_roll_mean,compressor_temp_c_roll_std,compressor_temp_c_trend,cargo_temp_deviation_c_roll_mean,cargo_temp_deviation_c_roll_std,cargo_temp_deviation_c_trend
0,1,0,1.818,140.81,65.07,0.121,0,1.81800,0.000000,0.0,140.810000,0.000000,0.0,65.070,0.000000,0.0,0.121000,0.000000,0.0
1,1,1,1.379,141.27,64.63,0.733,0,1.59850,0.310420,0.0,141.040000,0.325269,0.0,64.850,0.311127,0.0,0.427000,0.432749,0.0
2,1,2,1.633,142.92,65.09,0.300,0,1.61000,0.220402,0.0,141.666667,1.109519,0.0,64.930,0.260000,0.0,0.384667,0.314662,0.0
3,1,3,1.623,140.95,66.11,0.411,0,1.61325,0.180075,0.0,141.487500,0.974213,0.0,65.225,0.627030,0.0,0.391250,0.257258,0.0
4,1,4,1.483,143.21,67.88,0.058,0,1.58720,0.166473,0.0,141.832000,1.142462,0.0,65.756,1.305634,0.0,0.324600,0.268043,0.0


In [4]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

feat_df = feat_df.groupby("unit_id", group_keys=False).apply(lambda g: g.iloc[WINDOW:])
feat_df = feat_df.reset_index(drop=True)
print(feat_df.shape)

(180502, 19)


In [5]:
from sklearn.model_selection import GroupShuffleSplit

feature_cols = [c for c in feat_df.columns if c not in ["unit_id", "day", "failure_within_7d"]]

X = feat_df[feature_cols]
y = feat_df["failure_within_7d"]
groups = feat_df["unit_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print(f"Train: {len(X_train)} rows, {y_train.sum()} positive cases")
print(f"Test:  {len(X_test)} rows, {y_test.sum()} positive cases")

Train: 135691 rows, 357 positive cases
Test:  44811 rows, 133 positive cases


In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

rf = RandomForestClassifier(
    n_estimators=300,      # number of trees in the forest
    max_depth=10,          # limit tree depth to reduce overfitting
    class_weight="balanced",  # compensate for our ~0.3% positive rate
    random_state=42,
    n_jobs=-1              # use all CPU cores to train faster
)

rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
y_proba = rf.predict_proba(X_test)[:, 1]  # probability of class 1 (failing soon)

print(classification_report(y_test, y_pred, digits=3))
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba):.3f}")

              precision    recall  f1-score   support

           0      1.000     0.996     0.998     44678
           1      0.422     0.970     0.588       133

    accuracy                          0.996     44811
   macro avg      0.711     0.983     0.793     44811
weighted avg      0.998     0.996     0.997     44811

ROC-AUC: 0.999


In [7]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")

# For comparison: what accuracy would a "lazy" model get by just predicting 0 (no failure) for everything?
naive_accuracy = (y_test == 0).mean()
print(f"Naive 'always predict healthy' accuracy: {naive_accuracy:.4f}")

Accuracy: 0.9960
Naive 'always predict healthy' accuracy: 0.9970


In [8]:
export_df = feat_df.iloc[test_idx].copy()
export_df["predicted_probability"] = y_proba
export_df["predicted_label"] = y_pred

export_df.to_csv("thermoking_predictions_for_tableau.csv", index=False)
print(export_df.shape)
export_df.head()

(44811, 21)


,unit_id,day,vibration_mm_s,coolant_pressure_kpa,compressor_temp_c,cargo_temp_deviation_c,failure_within_7d,vibration_mm_s_roll_mean,vibration_mm_s_roll_std,vibration_mm_s_trend,...,coolant_pressure_kpa_roll_std,coolant_pressure_kpa_trend,compressor_temp_c_roll_mean,compressor_temp_c_roll_std,compressor_temp_c_trend,cargo_temp_deviation_c_roll_mean,cargo_temp_deviation_c_roll_std,cargo_temp_deviation_c_trend,predicted_probability,predicted_label
0,1,14,1.662,140.57,66.12,0.096,0,1.718857,0.128787,0.042,...,4.147419,-1.45,67.070000,1.272478,0.82,0.309000,0.325361,-0.310,0.0,0
1,1,15,1.712,144.85,63.82,0.396,0,1.696286,0.110418,-0.158,...,4.042115,-0.62,66.514286,1.717866,-3.89,0.363000,0.299343,0.378,0.0,0
2,1,16,1.530,139.91,66.27,0.495,0,1.643286,0.080863,-0.371,...,3.909100,1.62,66.238571,1.548778,-1.93,0.418000,0.279844,0.385,0.0,0
3,1,17,1.581,142.58,65.39,0.531,0,1.623429,0.075793,-0.139,...,3.938210,0.67,66.257143,1.535814,0.13,0.428429,0.282926,0.073,0.0,0
4,1,18,1.737,141.39,64.35,0.407,0,1.626429,0.080347,0.021,...,3.160400,-5.28,65.954286,1.688301,-2.12,0.436429,0.281158,0.056,0.0,0


In [9]:
import joblib
joblib.dump(rf, "rf_model.joblib")

['rf_model.joblib']

In [10]:
import sqlite3

# Build the export dataframe (test set rows + true labels + predictions)
export_df = feat_df.iloc[test_idx].copy()
export_df["predicted_probability"] = y_proba
export_df["predicted_label"] = y_pred

# Connect to (or create) a SQLite database file
conn = sqlite3.connect("thermoking.db")

# Write the dataframe into a table called "predictions"
export_df.to_sql("predictions", conn, if_exists="replace", index=False)

print("Rows written:", len(export_df))
conn.close()

Rows written: 44811


In [11]:
conn = sqlite3.connect("thermoking.db")

query = """
SELECT unit_id, COUNT(*) as total_days, 
       SUM(failure_within_7d) as true_failure_days,
       AVG(predicted_probability) as avg_risk_score,
       MAX(predicted_probability) as max_risk_score
FROM predictions
GROUP BY unit_id
ORDER BY max_risk_score DESC
LIMIT 10
"""

result = pd.read_sql(query, conn)
conn.close()
result

,unit_id,total_days,true_failure_days,avg_risk_score,max_risk_score
0,114,164,7,0.147935,0.998082
1,382,205,7,0.100949,0.998060
2,221,408,7,0.035508,0.998032
3,226,263,7,0.061967,0.998000
4,102,430,7,0.035143,0.997971
5,343,321,7,0.043541,0.997949
6,228,120,7,0.127667,0.997949
7,224,385,7,0.048562,0.997894
8,181,361,7,0.056305,0.997809
9,330,143,7,0.081430,0.997779


In [12]:
conn = sqlite3.connect("thermoking.db")

query2 = """
SELECT unit_id, COUNT(*) as total_days,
       SUM(failure_within_7d) as true_failure_days,
       AVG(predicted_probability) as avg_risk_score,
       MAX(predicted_probability) as max_risk_score
FROM predictions
GROUP BY unit_id
HAVING true_failure_days = 0
ORDER BY max_risk_score DESC
LIMIT 10
"""

healthy_check = pd.read_sql(query2, conn)
conn.close()
healthy_check

,unit_id,total_days,true_failure_days,avg_risk_score,max_risk_score
0,397,486,0,0.0,0.0
1,396,486,0,0.0,0.0
2,395,486,0,0.0,0.0
3,392,486,0,0.0,0.0
4,389,486,0,0.0,0.0
5,383,486,0,0.0,0.0
6,377,486,0,0.0,0.0
7,375,486,0,0.0,0.0
8,370,486,0,0.0,0.0
9,362,486,0,0.0,0.0


In [13]:
conn = sqlite3.connect("thermoking.db")

# Full daily-level data (for time-series / trend charts in Tableau)
daily_query = "SELECT * FROM predictions"
daily_df = pd.read_sql(daily_query, conn)
daily_df.to_csv("tableau_daily_predictions.csv", index=False)

# Per-unit summary (for a fleet risk leaderboard view)
summary_query = """
SELECT unit_id, COUNT(*) as total_days,
       SUM(failure_within_7d) as true_failure_days,
       AVG(predicted_probability) as avg_risk_score,
       MAX(predicted_probability) as max_risk_score
FROM predictions
GROUP BY unit_id
"""
summary_df = pd.read_sql(summary_query, conn)
summary_df.to_csv("tableau_unit_summary.csv", index=False)

conn.close()
print("Exported:", daily_df.shape, summary_df.shape)

Exported: (44811, 21) (100, 5)
